# Weather Router — Demo

This notebook serves as an interactive test client for the `weather_router` library, before a proper user interface is built.

## How it works

1. **Upload a GPX file** – the route is split into clusters based on speed and gradient
2. **Fetch weather data** – a 15-minute forecast is retrieved from [Open-Meteo](https://open-meteo.com/) for each cluster's midpoint
3. **Calculate score** – each segment is scored by wind (50 %), gusts (30 %), and precipitation (20 %), aggregated by distance
4. **Show map** – wind arrows indicate direction and speed; the route is colored by segment score (green → red)

## Score Scale

| Score | Meaning |
|-------|---------|
| **0.5 – 1.0** | Good conditions – tailwind, no gusts, dry |
| **0.0 – 0.5** | Acceptable conditions – light crosswind or headwind |
| **−0.5 – 0.0** | Difficult conditions – stronger headwind or rain |
| **−1.0 – −0.5** | Bad conditions – strong gusts or heavy rain |

The score is dimensionless and distance-weighted: short segments with poor weather pull the overall score down less than long ones.

## Interactive Analysis

Upload a GPX file, set your average speed and start time, then click **Analyze Route**.

In [ ]:
from datetime import datetime
import math
import ipywidgets as widgets
from IPython.display import display
import base64
import folium
from folium.features import ColorLine
from app.services.gpx_parser import get_clustered_route, parse_gpx
from app.services.weather import get_weather_for_route
from app.services.route_scorer import score_segment, MAX_GUST_SPEED_KM_H, MAX_GUST_DELTA_KM_H, MAX_PRECIPITATION_MM_H

file_upload = widgets.FileUpload(
    accept=".gpx",
    multiple=False,
    description="GPX File",
    style={"button_color": "lightblue"},
)

speed_slider = widgets.FloatSlider(
    value=15.0,
    min=1.0,
    max=60.0,
    step=0.5,
    description="Avg Speed (km/h):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="450px"),
)

start_time_picker = widgets.NaiveDatetimePicker(
    value=datetime.now().replace(second=0, microsecond=0),
    description="Start Time:",
    style={"description_width": "initial"},
)

submit_button = widgets.Button(
    description="Analyze Route",
    button_style="primary",
    icon="play",
    layout=widgets.Layout(margin="8px 0"),
)

score_output = widgets.Output()
map_widget = widgets.HTML(value="", layout=widgets.Layout(width="100%", height="500px"))


def _score_to_color(score: float) -> str:
    """Interpolates from red (−1) through yellow (0) to green (1)."""
    t = (score + 1.0) / 2.0  # 0..1
    if t < 0.5:
        r, g = 220, int(t * 2 * 200)
    else:
        r, g = int((1 - t) * 2 * 220), 180
    return f"#{r:02x}{g:02x}32"


def _score_label(score: float) -> tuple[str, str]:
    """Returns (label, color) for a given score."""
    if score >= 0.5:
        return "Good conditions", "#2e7d32"
    elif score >= 0.0:
        return "Acceptable conditions", "#f9a825"
    elif score >= -0.5:
        return "Difficult conditions", "#e65100"
    else:
        return "Bad conditions", "#b71c1c"


def _wind_components(snap) -> tuple[float, float, str]:
    """Returns (wind_score, wind_speed_km_h, alignment_label) for a snapshot."""
    speed = snap.wind_speed_km_h
    bearing = snap.cluster.mean_bearing
    wind_dir = snap.wind_direction_deg

    rad_b = math.radians(bearing)
    bx, by = math.sin(rad_b), math.cos(rad_b)
    rad_w = math.radians((wind_dir + 180) % 360)
    wx, wy = math.sin(rad_w), math.cos(rad_w)
    dot = bx * wx + by * wy

    if dot > 0.5:
        label, max_spd = "tailwind", 50.0
    elif dot < -0.5:
        label, max_spd = "headwind", 60.0
    else:
        label, max_spd = "crosswind", 50.0

    wind_score = max(-1.0, min(1.0, dot * (speed / max_spd)))
    return wind_score, speed, label


def _weighted_breakdown(snapshots, total_distance_m):
    """Compute distance-weighted averages of wind, gust, and rain score components."""
    w_wind = w_gust = w_rain = 0.0
    avg_speed = avg_gust = avg_delta = avg_precip = 0.0
    wind_labels: dict[str, float] = {}

    for snap in snapshots:
        dist = snap.cluster.total_distance_m
        weight = dist / total_distance_m

        wind_score, speed, wind_label = _wind_components(snap)
        gust_score = -min(snap.wind_gusts_km_h / MAX_GUST_SPEED_KM_H, 1.0)
        precip_mm_h = snap.precipitation_mm_h * 4.0
        rain_score = -min(precip_mm_h / MAX_PRECIPITATION_MM_H, 1.0)
        delta = snap.wind_gusts_km_h - speed

        w_wind += wind_score * weight
        w_gust += gust_score * weight
        w_rain += rain_score * weight
        avg_speed += speed * weight
        avg_gust += snap.wind_gusts_km_h * weight
        avg_delta += delta * weight
        avg_precip += precip_mm_h * weight
        wind_labels[wind_label] = wind_labels.get(wind_label, 0.0) + weight

    dominant_wind = max(wind_labels, key=wind_labels.get)
    return w_wind, w_gust, w_rain, avg_speed, avg_gust, avg_delta, avg_precip, dominant_wind


def plot_route(snapshots, gpx_bytes, segment_scores):
    points = parse_gpx(gpx_bytes)
    lats = [p.lat for p in points]
    lons = [p.lon for p in points]
    center = [sum(lats) / len(lats), sum(lons) / len(lons)]

    m = folium.Map(location=center, zoom_start=11)

    # Color route segments by score
    for snap, seg_score in zip(snapshots, segment_scores):
        segs = snap.cluster.segments
        coords = [(segs[0].start.lat, segs[0].start.lon)] + [
            (s.end.lat, s.end.lon) for s in segs
        ]
        folium.PolyLine(
            coords,
            color=_score_to_color(seg_score),
            weight=5,
            opacity=0.85,
            tooltip=f"Score: {seg_score:.2f}",
        ).add_to(m)

    # Wind arrows
    for snap in snapshots:
        rp = snap.cluster.representative_point
        arrow_deg = (snap.wind_direction_deg + 180) % 360

        if snap.precipitation_mm_h >= 1.0:
            arrow_color = "#0000cc"
        elif snap.precipitation_mm_h >= 0.1:
            arrow_color = "#4444ff"
        else:
            arrow_color = "#888888"

        icon_html = f"""
        <div style="width:44px; text-align:center; line-height:1.1;">
          <div style="transform:rotate({arrow_deg:.0f}deg); font-size:20px; color:{arrow_color};">&#8593;</div>
          <div style="font-size:9px; color:#222;">{snap.wind_speed_km_h:.0f}&nbsp;km/h</div>
        </div>
        """
        popup_html = (
            f"<b>Time:</b> {snap.timestamp.strftime('%H:%M')}<br>"
            f"<b>Wind:</b> {snap.wind_speed_km_h:.1f} km/h<br>"
            f"<b>Gusts:</b> {snap.wind_gusts_km_h:.1f} km/h<br>"
            f"<b>Rain:</b> {snap.precipitation_mm_h:.2f} mm"
        )
        folium.Marker(
            location=[rp.lat, rp.lon],
            popup=folium.Popup(popup_html, max_width=220),
            icon=folium.DivIcon(html=icon_html, icon_size=(44, 44), icon_anchor=(22, 22)),
        ).add_to(m)

    map_html = m.get_root().render()
    encoded = base64.b64encode(map_html.encode()).decode()
    map_widget.value = f'<iframe src="data:text/html;base64,{encoded}" width="100%" height="500px" frameborder="0"></iframe>'


def on_submit(b):
    score_output.clear_output()
    map_widget.value = ""
    submit_button.disabled = True
    submit_button.description = "Analyzing..."

    try:
        with score_output:
            if not file_upload.value:
                print("Please upload a GPX file first.")
                return

            content = file_upload.value[0]["content"].tobytes()
            print("Segmenting route...")
            route_clusters = get_clustered_route(content, speed_slider.value, start_time_picker.value)
            n = len(route_clusters.clusters)
            dist_km = route_clusters.total_distance_m / 1000
            print(f"  → {n} segments, {dist_km:.1f} km total distance, starting at {start_time_picker.value.strftime('%Y-%m-%d %H:%M')} estimated arrival at {route_clusters.representative_points[-1].timestamp.strftime('%Y-%m-%d %H:%M')}")

            print("Fetching weather data...")
            snapshots = get_weather_for_route(route_clusters)
            print(f"  → {len(snapshots)} forecasts loaded")

            print("Calculating score...")
            segment_scores = [score_segment(s) for s in snapshots]
            total_score = sum(
                sc * s.cluster.total_distance_m
                for sc, s in zip(segment_scores, snapshots)
            ) / route_clusters.total_distance_m

            w_wind, w_gust, w_rain, avg_speed, avg_gust, avg_delta, avg_precip, dominant_wind = (
                _weighted_breakdown(snapshots, route_clusters.total_distance_m)
            )

            label, color = _score_label(total_score)
            print(
                f"\n{'─'*40}\n"
                f"  Overall Score: {total_score:+.2f}  |  {label}\n"
                f"{'─'*40}\n"
                f"  Why:\n"
                f"  Wind  (50 %): {w_wind * 0.5:+.2f}  →  avg {avg_speed:.1f} km/h, mostly {dominant_wind}\n"
                f"  Gusts (30 %): {w_gust * 0.3:+.2f}  →  avg {avg_gust:.1f} km/h (Δ {avg_delta:+.1f} km/h vs wind, limit Δ{MAX_GUST_DELTA_KM_H:.0f})\n"
                f"  Rain  (20 %): {w_rain * 0.2:+.2f}  →  avg {avg_precip:.2f} mm/h\n"
                f"{'─'*40}"
            )

        plot_route(snapshots, content, segment_scores)

    except Exception as e:
        with score_output:
            print(f"\nError: {e}")
    finally:
        submit_button.disabled = False
        submit_button.description = "Analyze Route"


submit_button._click_handlers.callbacks.clear()
submit_button.on_click(on_submit)


In [ ]:
display(
    file_upload,
    speed_slider,
    start_time_picker,
    submit_button,
    score_output,
    map_widget,
)
